In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms

from training.train_utils import set_seed, Model, VAE, CelebA
from training.imm_utils import ResidualDenoiser
from training.ddpm_utils import DDPM

def compute_mmd(x, y):
    # Simple Gaussian kernel MMD
    def gaussian_kernel(a, b):
        dim = a.shape[1]
        dist = torch.cdist(a, b, p=2)**2
        return torch.exp(-dist / dim)
    
    xx = gaussian_kernel(x, x)
    yy = gaussian_kernel(y, y)
    xy = gaussian_kernel(x, y)
    return xx.mean() + yy.mean() - 2*xy.mean()

def convert_to_display(samples):
    return samples

def show_reconstructions(model, dataloader, device, n=8):
    model.eval()
    with torch.no_grad():
        imgs, _ = next(iter(dataloader))
        imgs = imgs[:n].to(device)
        _, recon = model(imgs)
        
        imgs = imgs.cpu().numpy().transpose(0, 2, 3, 1) * 0.5 + 0.5
        recon = recon.cpu().numpy().transpose(0, 2, 3, 1) * 0.5 + 0.5
        
        fig, axs = plt.subplots(2, n, figsize=(n*2, 4))
        for i in range(n):
            axs[0, i].imshow(np.clip(imgs[i], 0, 1))
            axs[0, i].axis('off')
            axs[1, i].imshow(np.clip(recon[i], 0, 1))
            axs[1, i].axis('off')
        plt.show()


In [ ]:
def train(dataloader, z_dim=64, n_epochs=10, use_cuda=True, print_every=100, plot_every=500):
    device = torch.device("cuda" if use_cuda and torch.cuda.is_available() else "cpu")
    print(f"Training on {device}...")

    # 1) Instantiate DDPM (Denoising network + Schedule)
    # We reuse the ResidualDenoiser structure, but now it will be conditioned on t inside DDPM wrapper
    # Note: ResidualDenoiser takes input_dim. DDPM wrapper concats time embedding.
    # z_dim + 64 (time_dim) = input_dim for the denoiser MLP
    time_dim = 64
    denoiser_network = ResidualDenoiser(
        input_dim=z_dim + time_dim,
        model_dim=1024,
        output_dim=z_dim,
        n_blocks=8
    )
    
    ddpm_model = DDPM(
        network=denoiser_network,
        z_dim=z_dim,
        n_steps=1000,
        device=device
    ).to(device)

    # 2) Instantiate the VAE
    vae_model = Model(3, 32, 3, 128, z_dim, 32, 32).to(device)

    # 3) Optimizers
    optimizer_vae = torch.optim.Adam(vae_model.parameters(), lr=1e-3)
    optimizer_ddpm = torch.optim.Adam(ddpm_model.parameters(), lr=1e-4)

    step_idx = 0
    for epoch in range(n_epochs):
        vae_model.train()
        ddpm_model.train()

        for images, _labels in dataloader:
            step_idx += 1
            x = images.to(device)

            # --- VAE Forward ---
            z, x_recon = vae_model(x)
            z_det = z.detach() # Stop gradients from DDPM flowing back into VAE encoder if we want separate training
            # Actually, standard LDM trains VAE first, then DM.
            # But here we are doing joint training.
            # If we want the VAE to shape its latent space to be friendly to DDPM, we might want gradients.
            # However, DDPM objective is to DENOISE. It doesn't inherently push z to be Gaussian.
            # So we MUST use the MMD regularization on z to ensure it stays Standard Normal,
            # because DDPM assumes the starting noise x_T is Normal.

            # (1) Reconstruction Loss
            recon_loss = F.mse_loss(x_recon, x, reduction='mean')

            # (2) Regularization (MMD to N(0,1))
            # Sample true N(0,1) 
            true_samples = torch.randn_like(z)
            mmd_loss = compute_mmd(true_samples, z)

            # (3) DDPM Loss
            # Train DDPM to model the distribution of z_det
            ddpm_loss = ddpm_model.compute_loss(z_det)

            # Total Losses
            loss_vae = recon_loss + 0.1 * mmd_loss # Weight MMD to keep latent space behaved
            loss_ddpm = ddpm_loss

            # Updates
            optimizer_vae.zero_grad()
            loss_vae.backward()
            optimizer_vae.step()

            optimizer_ddpm.zero_grad()
            loss_ddpm.backward()
            optimizer_ddpm.step()

            if step_idx % print_every == 0:
                print(f"[Epoch {epoch+1}/{n_epochs} Step {step_idx}] Recon={recon_loss.item():.5f} MMD={mmd_loss.item():.5f} DDPM={ddpm_loss.item():.5f}")

            if step_idx % plot_every == 0:
                vae_model.eval()
                ddpm_model.eval()
                
                # 1. Show VAE Reconstructions
                print("Reconstructions:")
                show_reconstructions(vae_model, dataloader, device, n=8)
                
                # 2. Show DDPM Samples
                print("DDPM Samples:")
                with torch.no_grad():
                    n_vis = 16
                    # Sample z from DDPM
                    z_gen = ddpm_model.sample(n_vis)
                    # Decode
                    samples = vae_model.decoder(z_gen)
                    samples = samples.cpu().permute(0, 2, 3, 1).numpy() * 0.5 + 0.5
                    
                    fig, axs = plt.subplots(2, 8, figsize=(16, 4))
                    for i in range(n_vis):
                        row = i // 8
                        col = i % 8
                        axs[row, col].imshow(np.clip(samples[i], 0, 1))
                        axs[row, col].axis('off')
                    plt.show()
                
                vae_model.train()
                ddpm_model.train()
    
    return vae_model, ddpm_model


In [ ]:
if __name__ == "__main__":
    set_seed(42)
    batch_size = 128
    image_size = 32
    
    celeba_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    
    celeba_train = DataLoader(
        CelebA("./data/CelebA", split='train', download=True, transform=celeba_transform),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=True
    )
    
    vae_model, ddpm_model = train(
        dataloader=celeba_train,
        z_dim=64,
        n_epochs=80,
        use_cuda=True,
        print_every=100,
        plot_every=1000
    )
    
    # Save models
    torch.save(vae_model.state_dict(), "vae_model_ddpm.pth")
    torch.save(ddpm_model.state_dict(), "ddpm_model.pth")
